### **Lenet training**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.Grayscale(num_output_channels=1)
])

# Load dataset
train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=8)
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True, num_workers=8)

writer = SummaryWriter(log_dir="runs/LeNet")


if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

# Move model to device
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1,out_channels= 6,kernel_size= 5)
        self.pool = nn.AvgPool2d(kernel_size=2,stride=2)
        self.conv2 = nn.Conv2d(in_channels=6,out_channels= 16,kernel_size= 5)
        self.conv3 = nn.Conv2d(in_channels=16,out_channels= 120,kernel_size= 5)
        self.fc1=nn.Linear(120,84)
        self.fc2=nn.Linear(84,7)
    def forward(self,X):
        X=torch.tanh(self.conv1(X))
        X=self.pool(X)
        X=torch.tanh(self.conv2(X))
        X=self.pool(X)
        X=torch.tanh(self.conv3(X))
        X = X.view(-1, 120)
        X=torch.tanh(self.fc1(X))
        X=self.fc2(X)
        return X
model=LeNet()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)

# Save the model
torch.save(model.state_dict(), 'LeNet_model.pth')

writer.close()


GPU is not available
Epoch 1/20: Loss: 759.1311, Accuracy: 0.3306, Precision: 0.2654, Recall: 0.2528, F1 score: 0.2295


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 184.8349, Accuracy: 0.3639, Precision: 0.2757, Recall: 0.2911, F1: 0.2728


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2/20: Loss: 727.8987, Accuracy: 0.3611, Precision: 0.2760, Recall: 0.2838, F1 score: 0.2652
Test: Loss: 180.1088, Accuracy: 0.3777, Precision: 0.3092, Recall: 0.2950, F1: 0.2760
Epoch 3/20: Loss: 709.3063, Accuracy: 0.3848, Precision: 0.3008, Recall: 0.3039, F1 score: 0.2897


...

Test: Loss: 172.8372, Accuracy: 0.4021, Precision: 0.3210, Recall: 0.3252, F1: 0.3128
Epoch 5/20: Loss: 676.7653, Accuracy: 0.4144, Precision: 0.3987, Recall: 0.3313, F1 score: 0.3207


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 171.5507, Accuracy: 0.4085, Precision: 0.3337, Recall: 0.3206, F1: 0.3089
Epoch 6/20: Loss: 664.2215, Accuracy: 0.4267, Precision: 0.3398, Recall: 0.3418, F1 score: 0.3322


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 166.1161, Accuracy: 0.4250, Precision: 0.3406, Recall: 0.3397, F1: 0.3331
Epoch 7/20: Loss: 653.7989, Accuracy: 0.4372, Precision: 0.4388, Recall: 0.3547, F1 score: 0.3475
Test: Loss: 165.8674, Accuracy: 0.4338, Precision: 0.4872, Recall: 0.3484, F1: 0.3333
Epoch 8/20: Loss: 631.6453, Accuracy: 0.4594, Precision: 0.4950, Recall: 0.3756, F1 score: 0.3716
Test: Loss: 161.6978, Accuracy: 0.4436, Precision: 0.4979, Recall: 0.3614, F1: 0.3554
Epoch 9/20: Loss: 627.8006, Accuracy: 0.4621, Precision: 0.4968, Recall: 0.3783, F1 score: 0.3748
Test: Loss: 162.0206, Accuracy: 0.4422, Precision: 0.5005, Recall: 0.3598, F1: 0.3566
Epoch 10/20: Loss: 624.7483, Accuracy: 0.4653, Precision: 0.5106, Recall: 0.3810, F1 score: 0.3795
Test: Loss: 161.6745, Accuracy: 0.4419, Precision: 0.4975, Recall: 0.3634, F1: 0.3614
Epoch 11/20: Loss: 623.2541, Accuracy: 0.4655, Precision: 0.4955, Recall: 0.3833, F1 score: 0.3820
Test: Loss: 160.8075, Accuracy: 0.4468, Precision: 0.4641, Recall: 0.3660, F1:

### **Alexnet training**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load dataset
train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=8)
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True, num_workers=8)

writer = SummaryWriter(log_dir="runs/alexnet")

model = models.alexnet(pretrained=True)
for par in model.parameters():
    par.requires_grad=False

model.classifier[6] = nn.Linear(model.classifier[6].in_features, 7)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

# Move model to device

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)

# Save the model
torch.save(model.state_dict(), 'Alexnet_model.pth')

writer.close()


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


GPU is not available
Epoch 1/20: Loss: 737.8120, Accuracy: 0.3558, Precision: 0.2927, Recall: 0.2914, F1 score: 0.2853


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 171.5725, Accuracy: 0.4057, Precision: 0.3420, Recall: 0.3335, F1: 0.3139
Epoch 2/20: Loss: 714.0134, Accuracy: 0.3836, Precision: 0.3487, Recall: 0.3188, F1 score: 0.3159
Test: Loss: 169.2997, Accuracy: 0.4172, Precision: 0.4063, Recall: 0.3426, F1: 0.3362
Epoch 3/20: Loss: 707.3113, Accuracy: 0.3892, Precision: 0.3562, Recall: 0.3255, F1 score: 0.3240


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 171.3169, Accuracy: 0.4028, Precision: 0.4072, Recall: 0.3070, F1: 0.2969
Epoch 4/20: Loss: 704.2613, Accuracy: 0.3890, Precision: 0.3655, Recall: 0.3287, F1 score: 0.3295
Test: Loss: 168.0311, Accuracy: 0.4213, Precision: 0.4570, Recall: 0.3361, F1: 0.3269
Epoch 5/20: Loss: 703.5613, Accuracy: 0.3923, Precision: 0.3530, Recall: 0.3301, F1 score: 0.3299
Test: Loss: 165.3459, Accuracy: 0.4365, Precision: 0.4568, Recall: 0.3583, F1: 0.3554
Epoch 6/20: Loss: 699.2250, Accuracy: 0.3959, Precision: 0.3575, Recall: 0.3346, F1 score: 0.3350
Test: Loss: 169.7812, Accuracy: 0.4100, Precision: 0.4609, Recall: 0.3332, F1: 0.3159
Epoch 7/20: Loss: 702.0279, Accuracy: 0.3915, Precision: 0.3612, Recall: 0.3325, F1 score: 0.3340
Test: Loss: 172.6130, Accuracy: 0.3958, Precision: 0.5008, Recall: 0.3562, F1: 0.3198
Epoch 8/20: Loss: 675.1259, Accuracy: 0.4171, Precision: 0.3935, Recall: 0.3555, F1 score: 0.3578
Test: Loss: 163.2727, Accuracy: 0.4394, Precision: 0.4580, Recall: 0.3730, F1: 0

### **Mobilenet training**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=8)
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True, num_workers=8)

writer = SummaryWriter(log_dir="runs/mobilenet")

model = models.mobilenet_v2(pretrained=True)
for par in model.parameters():
    par.requires_grad=False

model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 7)
)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

# Move model to device

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)

torch.save(model.state_dict(), 'mobilenet_model.pth')

writer.close()


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


GPU is not available


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/20: Loss: 736.8059, Accuracy: 0.3543, Precision: 0.1338, Recall: 0.1332, F1 score: 0.1300


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 170.9720, Accuracy: 0.4235, Precision: 0.3411, Recall: 0.3466, F1: 0.3309
Epoch 2/20: Loss: 702.8302, Accuracy: 0.3877, Precision: 0.3606, Recall: 0.3196, F1 score: 0.3138
Test: Loss: 172.1880, Accuracy: 0.4238, Precision: 0.4203, Recall: 0.3539, F1: 0.3319
Epoch 3/20: Loss: 700.2775, Accuracy: 0.3968, Precision: 0.3723, Recall: 0.3293, F1 score: 0.3262


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 174.0195, Accuracy: 0.4090, Precision: 0.3412, Recall: 0.3368, F1: 0.3035
Epoch 4/20: Loss: 700.4853, Accuracy: 0.3922, Precision: 0.3530, Recall: 0.3254, F1 score: 0.3221
Test: Loss: 168.0785, Accuracy: 0.4320, Precision: 0.5009, Recall: 0.3540, F1: 0.3495
Epoch 5/20: Loss: 695.0490, Accuracy: 0.4001, Precision: 0.3589, Recall: 0.3322, F1 score: 0.3277
Test: Loss: 169.6526, Accuracy: 0.4188, Precision: 0.5032, Recall: 0.3412, F1: 0.3392
Epoch 6/20: Loss: 695.7093, Accuracy: 0.4034, Precision: 0.3773, Recall: 0.3362, F1 score: 0.3330


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 170.0772, Accuracy: 0.4202, Precision: 0.3670, Recall: 0.3369, F1: 0.3385
Epoch 7/20: Loss: 695.1504, Accuracy: 0.3992, Precision: 0.3572, Recall: 0.3336, F1 score: 0.3311
Test: Loss: 168.9483, Accuracy: 0.4305, Precision: 0.4357, Recall: 0.3601, F1: 0.3375
Epoch 8/20: Loss: 680.0543, Accuracy: 0.4126, Precision: 0.4044, Recall: 0.3450, F1 score: 0.3422
Test: Loss: 167.2369, Accuracy: 0.4356, Precision: 0.4179, Recall: 0.3668, F1: 0.3624
Epoch 9/20: Loss: 674.8548, Accuracy: 0.4177, Precision: 0.4066, Recall: 0.3491, F1 score: 0.3466
Test: Loss: 165.8842, Accuracy: 0.4418, Precision: 0.4046, Recall: 0.3627, F1: 0.3555
Epoch 10/20: Loss: 678.4659, Accuracy: 0.4150, Precision: 0.3838, Recall: 0.3455, F1 score: 0.3416
Test: Loss: 166.6551, Accuracy: 0.4299, Precision: 0.3859, Recall: 0.3504, F1: 0.3389
Epoch 11/20: Loss: 675.1721, Accuracy: 0.4203, Precision: 0.3880, Recall: 0.3502, F1 score: 0.3464
Test: Loss: 165.5969, Accuracy: 0.4348, Precision: 0.4093, Recall: 0.3621, F1:

### **Custom CNN training**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=8)
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True, num_workers=8)

writer = SummaryWriter(log_dir="runs/EmotionCNN")

class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(EmotionCNN, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Flatten(),

            nn.Linear(256 * 6 * 6, 512), 
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.model(x)
model = EmotionCNN(num_classes=7)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")


model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)

# Save the model
torch.save(model.state_dict(), 'EmotionCNN_model.pth')

writer.close()


GPU is not available
Epoch 1/20: Loss: 812.0343, Accuracy: 0.3022, Precision: 0.2257, Recall: 0.2338, F1 score: 0.2188


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 175.1008, Accuracy: 0.3955, Precision: 0.3166, Recall: 0.3066, F1: 0.2914
Epoch 2/20: Loss: 679.7110, Accuracy: 0.4061, Precision: 0.3256, Recall: 0.3240, F1 score: 0.3058


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 160.9849, Accuracy: 0.4473, Precision: 0.3648, Recall: 0.3526, F1: 0.3301
Epoch 3/20: Loss: 622.9412, Accuracy: 0.4629, Precision: 0.3899, Recall: 0.3751, F1 score: 0.3650


...

Test: Loss: 144.8381, Accuracy: 0.5043, Precision: 0.4230, Recall: 0.4086, F1: 0.4045
Epoch 4/20: Loss: 583.1155, Accuracy: 0.5032, Precision: 0.4300, Recall: 0.4122, F1 score: 0.4038


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 139.6303, Accuracy: 0.5283, Precision: 0.4421, Recall: 0.4430, F1: 0.4346
Epoch 5/20: Loss: 558.9567, Accuracy: 0.5304, Precision: 0.4703, Recall: 0.4414, F1 score: 0.4382


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 134.0570, Accuracy: 0.5432, Precision: 0.4476, Recall: 0.4475, F1: 0.4302
Epoch 6/20: Loss: 536.8840, Accuracy: 0.5466, Precision: 0.4821, Recall: 0.4566, F1 score: 0.4535
Test: Loss: 130.7768, Accuracy: 0.5573, Precision: 0.5962, Recall: 0.4673, F1: 0.4622
Epoch 7/20: Loss: 521.8498, Accuracy: 0.5614, Precision: 0.5132, Recall: 0.4725, F1 score: 0.4722
Test: Loss: 127.0871, Accuracy: 0.5674, Precision: 0.5406, Recall: 0.4709, F1: 0.4577
Epoch 8/20: Loss: 479.9588, Accuracy: 0.5966, Precision: 0.5322, Recall: 0.5012, F1 score: 0.4981
Test: Loss: 119.3188, Accuracy: 0.5967, Precision: 0.5846, Recall: 0.5004, F1: 0.4927
Epoch 9/20: Loss: 461.1503, Accuracy: 0.6129, Precision: 0.5845, Recall: 0.5249, F1 score: 0.5298
Test: Loss: 118.1299, Accuracy: 0.6052, Precision: 0.5845, Recall: 0.5127, F1: 0.5090
Epoch 10/20: Loss: 450.9618, Accuracy: 0.6190, Precision: 0.5874, Recall: 0.5324, F1 score: 0.5377
Test: Loss: 116.9587, Accuracy: 0.6128, Precision: 0.6187, Recall: 0.5253, F1: 

### **Lenet training after using facial landmarks**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import dlib
import cv2
import numpy as np

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')
def extract_landmarks(img):
    image=img
    image_np=image.resize((96,96))
    image_np = np.array(img)  
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    faces = detector(gray)

    if len(faces) == 0:
        return img 

    for face in faces:
        landmarks = predictor(gray, face)
        for n in range(68):
            x, y = landmarks.part(n).x, landmarks.part(n).y
            cv2.circle(image_np, (x, y), 2, (0, 255, 0), -1)
    image_np=cv2.resize(image_np,(32,32))
    return Image.fromarray(image_np) 

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Lambda(lambda img: extract_landmarks(img)),
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1



# Load dataset
train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True )
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

writer = SummaryWriter(log_dir="runs/LeNet_face_landmarks")


if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1,out_channels= 6,kernel_size= 5)
        self.pool = nn.AvgPool2d(kernel_size=2,stride=2)
        self.conv2 = nn.Conv2d(in_channels=6,out_channels= 16,kernel_size= 5)
        self.conv3 = nn.Conv2d(in_channels=16,out_channels= 120,kernel_size= 5)
        self.fc1=nn.Linear(120,84)
        self.fc2=nn.Linear(84,7)
    def forward(self,X):
        X=torch.tanh(self.conv1(X))
        X=self.pool(X)
        X=torch.tanh(self.conv2(X))
        X=self.pool(X)
        X=torch.tanh(self.conv3(X))
        X = X.view(-1, 120)
        X=torch.tanh(self.fc1(X))
        X=self.fc2(X)
        return X
model=LeNet()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)


writer.close()


GPU is not available
Epoch 1/20: Loss: 1523.3019, Accuracy: 0.3258, Precision: 0.2439, Recall: 0.2473, F1 score: 0.2234
Test: Loss: 371.9059, Accuracy: 0.3533, Precision: 0.2664, Recall: 0.2737, F1: 0.2427


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2/20: Loss: 1458.6276, Accuracy: 0.3617, Precision: 0.2766, Recall: 0.2825, F1 score: 0.2648


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 357.2828, Accuracy: 0.3801, Precision: 0.2973, Recall: 0.2990, F1: 0.2846


...

Epoch 3/20: Loss: 1410.4502, Accuracy: 0.3866, Precision: 0.2983, Recall: 0.3055, F1 score: 0.2908


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 348.6966, Accuracy: 0.4016, Precision: 0.3127, Recall: 0.3188, F1: 0.3023
Epoch 4/20: Loss: 1372.3833, Accuracy: 0.4058, Precision: 0.3642, Recall: 0.3233, F1 score: 0.3112


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 338.7009, Accuracy: 0.4217, Precision: 0.3432, Recall: 0.3308, F1: 0.3215
Epoch 5/20: Loss: 1340.9783, Accuracy: 0.4200, Precision: 0.3789, Recall: 0.3361, F1 score: 0.3255
Test: Loss: 334.3685, Accuracy: 0.4213, Precision: 0.3366, Recall: 0.3336, F1: 0.3232
Epoch 6/20: Loss: 1318.1576, Accuracy: 0.4295, Precision: 0.4305, Recall: 0.3469, F1 score: 0.3391
Test: Loss: 330.4211, Accuracy: 0.4296, Precision: 0.4070, Recall: 0.3527, F1: 0.3464
Epoch 7/20: Loss: 1299.2914, Accuracy: 0.4381, Precision: 0.4417, Recall: 0.3571, F1 score: 0.3525
Test: Loss: 327.3400, Accuracy: 0.4312, Precision: 0.3761, Recall: 0.3520, F1: 0.3480
Epoch 8/20: Loss: 1256.6081, Accuracy: 0.4616, Precision: 0.4647, Recall: 0.3810, F1 score: 0.3804
Test: Loss: 322.5764, Accuracy: 0.4455, Precision: 0.4714, Recall: 0.3638, F1: 0.3598
Epoch 9/20: Loss: 1242.0804, Accuracy: 0.4662, Precision: 0.4872, Recall: 0.3869, F1 score: 0.3886
Test: Loss: 322.5683, Accuracy: 0.4443, Precision: 0.4736, Recall: 0.3655, 

### **Alexnet training after using facial landmarks**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import dlib
import cv2
import numpy as np
from PIL import Image
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')

def extract_landmarks(img):
    # Ensure image is in RGB format
    img = img.convert('RGB')  
    image_np = np.array(img) 
    
    gray_image = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    
    faces = detector(gray_image)  
    
    if len(faces) == 0: 
        return img
    
    for face in faces:
        landmarks = predictor(gray_image, face)
        for n in range(0, 68):  
            x = landmarks.part(n).x
            y = landmarks.part(n).y
            cv2.circle(image_np, (x, y), 2, (0, 255, 0), -1)  
    
    return Image.fromarray(image_np) 


def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Lambda(lambda img: extract_landmarks(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True, num_workers=0)

writer = SummaryWriter(log_dir="runs/alexnet_facial_landmarks")

model = models.alexnet(pretrained=True)
for par in model.parameters():
    par.requires_grad=False

model.classifier[6] = nn.Linear(model.classifier[6].in_features, 7)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

# Move model to device

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)

writer.close()


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


GPU is not available
Epoch 1/20: Loss: 781.4595, Accuracy: 0.3252, Precision: 0.2621, Recall: 0.2629, F1 score: 0.2585
Test: Loss: 184.2831, Accuracy: 0.3682, Precision: 0.3504, Recall: 0.2890, F1: 0.2853
Epoch 2/20: Loss: 756.6692, Accuracy: 0.3558, Precision: 0.3037, Recall: 0.2924, F1 score: 0.2896
Test: Loss: 175.2908, Accuracy: 0.4143, Precision: 0.3581, Recall: 0.3515, F1: 0.3352
Epoch 3/20: Loss: 756.3760, Accuracy: 0.3585, Precision: 0.3096, Recall: 0.2970, F1 score: 0.2960


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 175.3667, Accuracy: 0.4142, Precision: 0.3276, Recall: 0.3393, F1: 0.3161
Epoch 4/20: Loss: 755.0383, Accuracy: 0.3550, Precision: 0.3025, Recall: 0.2923, F1 score: 0.2901
Test: Loss: 173.7308, Accuracy: 0.4143, Precision: 0.3517, Recall: 0.3427, F1: 0.3333
Epoch 5/20: Loss: 755.3127, Accuracy: 0.3568, Precision: 0.3060, Recall: 0.2962, F1 score: 0.2953
Test: Loss: 175.2007, Accuracy: 0.4129, Precision: 0.3525, Recall: 0.3538, F1: 0.3427
Epoch 6/20: Loss: 755.1492, Accuracy: 0.3605, Precision: 0.3105, Recall: 0.3018, F1 score: 0.3011
Test: Loss: 171.9494, Accuracy: 0.4179, Precision: 0.3967, Recall: 0.3371, F1: 0.3327
Epoch 7/20: Loss: 753.0485, Accuracy: 0.3628, Precision: 0.3116, Recall: 0.3018, F1 score: 0.3009
Test: Loss: 178.5386, Accuracy: 0.3894, Precision: 0.3671, Recall: 0.3375, F1: 0.3348
Epoch 8/20: Loss: 713.3482, Accuracy: 0.3868, Precision: 0.3507, Recall: 0.3227, F1 score: 0.3220
Test: Loss: 168.4845, Accuracy: 0.4280, Precision: 0.3728, Recall: 0.3531, F1: 0

### **Mobilenet_v2 training after using facial landmarks**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import dlib
import cv2
import numpy as np
from PIL import Image

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')

def extract_landmarks(img):
    img = img.convert('RGB') 
    image_np = np.array(img)  
    
    gray_image = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    
    faces = detector(gray_image)  
    
    if len(faces) == 0:  
        return img
    
    for face in faces:
        landmarks = predictor(gray_image, face)
        for n in range(0, 68):  
            x = landmarks.part(n).x
            y = landmarks.part(n).y
            cv2.circle(image_np, (x, y), 2, (0, 255, 0), -1)  
    
    return Image.fromarray(image_np)  

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Lambda(lambda img: extract_landmarks(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
train_data = datasets.ImageFolder('train', transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
test_data = datasets.ImageFolder('test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=True, num_workers=0)

writer = SummaryWriter(log_dir="runs/mobilenetv2_facial_landmarks")

model = models.mobilenet_v2(pretrained=True)
for par in model.parameters():
    par.requires_grad=False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 7)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

# Move model to device
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    scheduler.step()
    
    # Validation
    model.eval()
    val_labels = []
    val_preds = []
    val_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_labels.extend(labels.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
    val_accuracy, val_precision, val_recall, val_f1 = calculate_metrics(val_labels, val_preds)
    print(f"Test: Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    writer.add_scalar("Test Loss", val_loss, epoch)
    writer.add_scalar("Test Accuracy", val_accuracy, epoch)
    writer.add_scalar("Test Precision", val_precision, epoch)
    writer.add_scalar("Test Recall", val_recall, epoch)
    writer.add_scalar("Test F1 Score", val_f1, epoch)

writer.close()


GPU is not available
Epoch 1/20: Loss: 734.6984, Accuracy: 0.3566, Precision: 0.2867, Recall: 0.2868, F1 score: 0.2809


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 172.3087, Accuracy: 0.4055, Precision: 0.3322, Recall: 0.3328, F1: 0.3108
Epoch 2/20: Loss: 703.6034, Accuracy: 0.3902, Precision: 0.3318, Recall: 0.3196, F1 score: 0.3130


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 169.7456, Accuracy: 0.4195, Precision: 0.3388, Recall: 0.3469, F1: 0.3305
Epoch 3/20: Loss: 700.0489, Accuracy: 0.3926, Precision: 0.3493, Recall: 0.3239, F1 score: 0.3191
Test: Loss: 172.9739, Accuracy: 0.4099, Precision: 0.3799, Recall: 0.3342, F1: 0.2984
Epoch 4/20: Loss: 697.7348, Accuracy: 0.3994, Precision: 0.3739, Recall: 0.3319, F1 score: 0.3306
Test: Loss: 172.5409, Accuracy: 0.4104, Precision: 0.4252, Recall: 0.3380, F1: 0.3134
Epoch 5/20: Loss: 696.2162, Accuracy: 0.3995, Precision: 0.3498, Recall: 0.3314, F1 score: 0.3279
Test: Loss: 168.4043, Accuracy: 0.4198, Precision: 0.4406, Recall: 0.3450, F1: 0.3259
Epoch 6/20: Loss: 695.9788, Accuracy: 0.3963, Precision: 0.3561, Recall: 0.3289, F1 score: 0.3271
Test: Loss: 169.4995, Accuracy: 0.4232, Precision: 0.4268, Recall: 0.3624, F1: 0.3524
Epoch 7/20: Loss: 694.6515, Accuracy: 0.4022, Precision: 0.3665, Recall: 0.3363, F1 score: 0.3349
Test: Loss: 167.6848, Accuracy: 0.4306, Precision: 0.3586, Recall: 0.3485, F1: 0

### **Custom CNN training after using facial landmarks**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import dlib
import cv2
import numpy as np
from PIL import Image

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')

def extract_landmarks(img):
    img = np.array(img)  
    
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) 
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)  
    faces = detector(gray)  
    
    if len(faces) == 0:
        return Image.fromarray(gray)
    
    for face in faces:
        landmarks = predictor(gray, face)
        for n in range(68):  
            x, y = landmarks.part(n).x, landmarks.part(n).y
            cv2.circle(gray, (x, y), 2, (255, 255, 255), -1) 
    
    return Image.fromarray(gray)  

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return accuracy, precision, recall, f1

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.Lambda(lambda img: extract_landmarks(img)),   
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
     
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), 
    transforms.Resize((48, 48)),
    transforms.Lambda(lambda img: extract_landmarks(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  
])

train_data = datasets.ImageFolder('train', transform=train_transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_data = datasets.ImageFolder('test', transform=test_transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

writer = SummaryWriter(log_dir="runs/EmotionCNN_with_landmarks")

class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(EmotionCNN, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),  

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),  

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),  

            nn.Flatten(),  
            nn.Linear(256 * 6 * 6, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.model(x)


model = EmotionCNN(num_classes=7)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU is not available")

model = model.to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.5, verbose=True)
patience = 4
best_loss = float('inf')
epochs_without_improvement = 0
best_model_path = 'best_EmotionCNN_model.pth'

num_epochs = 20
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    accuracy, precision, recall, f1 = calculate_metrics(all_labels, all_preds)
    
    print(f"Epoch {epoch}/{num_epochs}: Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 score: {f1:.4f}")
    writer.add_scalar("Training Loss", total_loss, epoch)
    writer.add_scalar("Accuracy", accuracy, epoch)
    writer.add_scalar("Precision", precision, epoch)
    writer.add_scalar("Recall", recall, epoch)
    writer.add_scalar("F1 Score", f1, epoch)
    
    model.eval()
    test_labels = []
    test_preds = []
    test_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            test_labels.extend(labels.cpu().numpy())
            test_preds.extend(preds.cpu().numpy())
    
    test_accuracy, test_precision, test_recall, test_f1 = calculate_metrics(test_labels, test_preds)
    print(f"Test: Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")
    writer.add_scalar("Test Loss", test_loss, epoch)
    writer.add_scalar("Test Accuracy", test_accuracy, epoch)
    writer.add_scalar("Test Precision", test_precision, epoch)
    writer.add_scalar("Test Recall", test_recall, epoch)
    writer.add_scalar("Test F1 Score", test_f1, epoch)
    
    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), 'best_model_emotion.pth')
        print(f"Best model saved at epoch {epoch}.")
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    
    # Early stopping
    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break
    

    scheduler.step(test_loss)

model.load_state


GPU is not available


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/20: Loss: 838.8110, Accuracy: 0.2886, Precision: 0.2124, Recall: 0.2163, F1 score: 0.1993


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 189.3481, Accuracy: 0.3782, Precision: 0.2771, Recall: 0.3100, F1: 0.2730
Best model saved at epoch 1.


...

c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 145.5347, Accuracy: 0.5933, Precision: 0.4911, Recall: 0.4972, F1: 0.4915
Best model saved at epoch 7.
Epoch 8/20: Loss: 605.2132, Accuracy: 0.5712, Precision: 0.4695, Recall: 0.4730, F1 score: 0.4673
Test: Loss: 145.3861, Accuracy: 0.5915, Precision: 0.6405, Recall: 0.4912, F1: 0.4865
Best model saved at epoch 8.
Epoch 9/20: Loss: 596.1760, Accuracy: 0.5809, Precision: 0.5640, Recall: 0.4835, F1 score: 0.4797


c:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test: Loss: 143.3666, Accuracy: 0.5997, Precision: 0.4895, Recall: 0.5025, F1: 0.4877
Best model saved at epoch 9.
Epoch 10/20: Loss: 583.5078, Accuracy: 0.5970, Precision: 0.5372, Recall: 0.4988, F1 score: 0.4964
Test: Loss: 141.8573, Accuracy: 0.6035, Precision: 0.6506, Recall: 0.5047, F1: 0.5066
Best model saved at epoch 10.
Epoch 11/20: Loss: 576.0927, Accuracy: 0.6050, Precision: 0.5661, Recall: 0.5068, F1 score: 0.5061
Test: Loss: 141.4310, Accuracy: 0.6042, Precision: 0.6568, Recall: 0.5008, F1: 0.5057
Best model saved at epoch 11.
Epoch 12/20: Loss: 566.7450, Accuracy: 0.6160, Precision: 0.5760, Recall: 0.5198, F1 score: 0.5219
Test: Loss: 138.1817, Accuracy: 0.6227, Precision: 0.6142, Recall: 0.5405, F1: 0.5468
Best model saved at epoch 12.
Epoch 13/20: Loss: 558.7067, Accuracy: 0.6265, Precision: 0.5876, Recall: 0.5324, F1 score: 0.5369
Test: Loss: 138.3131, Accuracy: 0.6227, Precision: 0.6201, Recall: 0.5349, F1: 0.5265
Epoch 14/20: Loss: 548.6581, Accuracy: 0.6405, Precisio

AttributeError: 'EmotionCNN' object has no attribute 'load_state'